# Batch Query Test Against Known-Pose Map

This notebook localizes test/query images against the known-pose reconstruction produced by `reconstruction_known_pose.ipynb`. Query images can come from any camera as long as their basenames exist in `metadata/poses.json`.

The map and query cameras do not need to share the same camera_name. Ground truth is matched by image basename.

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import copy
import json
import random
import shutil

import numpy as np
import pandas as pd
import pycolmap
import torch

from hloc import extract_features, match_features, pairs_from_retrieval
from hloc.localize_sfm import QueryLocalizer, pose_from_cluster
from hloc.utils.parsers import parse_retrieval

from simulation_pose_utils import (
    image_basename,
    load_json,
    load_pose_records,
    metadata_pose_center,
    pycolmap_transform_rt,
    quat_wxyz_to_rotmat,
)

# Dataset with query metadata.
dataset_root = Path('../datasets/test_dataset_108_24_may')
intrinsics_json = dataset_root / 'metadata/intrinsics_pinhole.json'

# Query image folder. Can contain any camera as long as basenames exist in poses.json.
query_source_dir = dataset_root / 'test'
query_glob = '*.png'
max_queries = 200
random_seed = 42

# Known-pose map bundle created by reconstruction_known_pose.ipynb.
map_bundle_root = Path('../outputs/train_dataset_585_24_may-bundle')
sfm_model_root = map_bundle_root / 'sfm'
db_features = map_bundle_root / 'features.h5'
db_global_features = map_bundle_root / 'global-feats-netvlad.h5'

# Query outputs.
results_dir = map_bundle_root / 'query_batch_results_v4'
query_cache_dir = map_bundle_root / 'query_batch_v4_cache'
results_dir.mkdir(parents=True, exist_ok=True)
query_cache_dir.mkdir(parents=True, exist_ok=True)

# Localization parameters.
num_loc = 10
max_error = 12
# FIX: Always overwrite so cached stale features from partial runs don't cause silent skips.
overwrite_query_features = True

feature_conf = copy.deepcopy(extract_features.confs['superpoint_max'])
retrieval_conf = extract_features.confs['netvlad']
matcher_conf = match_features.confs['superpoint+lightglue']

# FIX: Verify GPU availability before running the pipeline.
gpu_available = torch.cuda.is_available()
print(f'CUDA available: {gpu_available}')
if gpu_available:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Running on CPU will be significantly slower.')

# Windows/sandbox-safe HLoc execution: avoid multiprocessing DataLoader workers.
# Keep class-compatible with Kornia's DataLoader[Any] annotation.
_original_dataloader = torch.utils.data.DataLoader
class _SingleProcessDataLoader(_original_dataloader):
    @classmethod
    def __class_getitem__(cls, item):
        return cls

    def __init__(self, *args, **kwargs):
        kwargs['num_workers'] = 0
        # FIX: Only disable pin_memory when there is no GPU; pin_memory speeds up GPU transfers.
        kwargs['pin_memory'] = torch.cuda.is_available()
        super().__init__(*args, **kwargs)
torch.utils.data.DataLoader = _SingleProcessDataLoader

for path in [query_source_dir, sfm_model_root, db_features, db_global_features]:
    if not path.exists():
        raise FileNotFoundError(path)

intrinsics_cfg = load_json(intrinsics_json)['cameras'][0]
print(f'Dataset: {dataset_root}')
print(f'Query dir: {query_source_dir}')
print(f'Map bundle: {map_bundle_root}')
print(f'SfM model: {sfm_model_root}')
print(f'Intrinsics: {intrinsics_cfg["model"]} {intrinsics_cfg["width"]}x{intrinsics_cfg["height"]} params={intrinsics_cfg["params"]}')


Dataset: ..\datasets\test_dataset_108_24_may
Query dir: ..\datasets\test_dataset_108_24_may\test
Map bundle: ..\outputs\train_dataset_585_24_may-bundle
SfM model: ..\outputs\train_dataset_585_24_may-bundle\sfm
Intrinsics: PINHOLE 1920x1080 params=[879.6779270567265, 879.6779270567265, 960.0, 540.0]


## 2. Load Map and Query Ground Truth

In [6]:
model = pycolmap.Reconstruction(sfm_model_root)
print(model.summary())

all_pose_records, pose_by_name, _ = load_pose_records(dataset_root, camera_names=None)
query_paths_all = sorted(p for p in query_source_dir.glob(query_glob) if p.is_file())
if max_queries is not None and max_queries < len(query_paths_all):
    rng = random.Random(random_seed)
    query_paths = sorted(rng.sample(query_paths_all, max_queries))
else:
    query_paths = query_paths_all

missing_metadata = [p.name for p in query_paths if p.name not in pose_by_name]
if missing_metadata:
    raise ValueError(f'{len(missing_metadata)} query images have no poses.json metadata. Examples: {missing_metadata[:10]}')

query_camera_distribution = {}
for p in query_paths:
    cam = pose_by_name[p.name]['camera_name']
    query_camera_distribution[cam] = query_camera_distribution.get(cam, 0) + 1

references = sorted(image.name for image in model.images.values())
print(f'All query images found: {len(query_paths_all)}')
print(f'Queries selected: {len(query_paths)}')
print(f'Random seed: {random_seed if max_queries is not None else None}')
print(f'Query camera distribution: {query_camera_distribution}')
print(f'Reference images in map: {len(references)}')

Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 585
	num_reg_frames = 585
	num_images = 585
	num_points3D = 190906
	num_observations = 755022
	mean_track_length = 3.95494
	mean_observations_per_image = 1290.64
	mean_reprojection_error = 1.29475
All query images found: 108
Queries selected: 108
Random seed: 42
Query camera distribution: {'front': 108}
Reference images in map: 585


## 3. Helper Functions

In [7]:
def extract_on_cpu(conf, image_root, image_list, feature_path, overwrite=True):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    original = torch.cuda.is_available
    torch.cuda.is_available = lambda: False
    try:
        return extract_features.main(
            conf,
            image_root,
            image_list=image_list,
            feature_path=feature_path,
            overwrite=overwrite,
        )
    finally:
        torch.cuda.is_available = original


def resolve_reference_ids(model, names):
    ref_ids = []
    missing = []
    for name in names:
        image = model.find_image_with_name(name)
        if image is None:
            basename = image_basename(name)
            for candidate in model.images.values():
                if image_basename(candidate.name) == basename:
                    image = candidate
                    break
        if image is None:
            missing.append(name)
        else:
            ref_ids.append(image.image_id)
    if missing:
        raise ValueError('Retrieved images not registered in model: ' + ', '.join(missing))
    return ref_ids


CARLA_TO_COLMAP_S = np.array([
    [0.0, 1.0, 0.0],
    [0.0, 0.0, -1.0],
    [1.0, 0.0, 0.0],
], dtype=np.float64)
COLMAP_TO_CARLA_S = np.linalg.inv(CARLA_TO_COLMAP_S)


def wrap_angle_deg(angle):
    return ((float(angle) + 180.0) % 360.0) - 180.0


def heading_error_deg(est_heading, gt_heading):
    return abs(wrap_angle_deg(float(est_heading) - float(gt_heading)))


def colmap_world_to_camera_to_carla_yaw_deg(r_wc_rh):
    # PnP/pycolmap returns COLMAP right-handed world-to-camera rotation.
    # Convert it back to CARLA left-handed camera-to-world rotation before extracting yaw.
    r_cw_rh = np.asarray(r_wc_rh, dtype=np.float64).T
    r_cw_lh = COLMAP_TO_CARLA_S @ r_cw_rh @ CARLA_TO_COLMAP_S
    return wrap_angle_deg(np.degrees(np.arctan2(r_cw_lh[1, 0], r_cw_lh[0, 0])))


def metadata_record_to_carla_yaw_deg(record):
    r_wc_rh = quat_wxyz_to_rotmat([record['qw'], record['qx'], record['qy'], record['qz']])
    return colmap_world_to_camera_to_carla_yaw_deg(r_wc_rh)


metadata_yaw_errors = []
for query_path in query_paths:
    rec = pose_by_name[query_path.name]
    metadata_yaw_errors.append(heading_error_deg(metadata_record_to_carla_yaw_deg(rec), rec['yaw_deg']))
metadata_yaw_errors = np.array(metadata_yaw_errors, dtype=np.float64)
print('Metadata quaternion -> CARLA yaw sanity check')
print(f'  count: {len(metadata_yaw_errors)}')
print(f'  mean error: {metadata_yaw_errors.mean():.9f} deg')
print(f'  max error: {metadata_yaw_errors.max():.9f} deg')
if metadata_yaw_errors.max() >= 1e-3:
    raise AssertionError('Metadata quaternion to CARLA yaw conversion sanity check failed.')


localizer_conf = {
    'estimation': {'ransac': {'max_error': max_error}},
    'refinement': {'refine_focal_length': False, 'refine_extra_params': False},
}
localizer = QueryLocalizer(model, localizer_conf)
camera = pycolmap.Camera(
    model=intrinsics_cfg['model'],
    width=int(intrinsics_cfg['width']),
    height=int(intrinsics_cfg['height']),
    params=np.array(intrinsics_cfg['params'], dtype=float),
)

Metadata quaternion -> CARLA yaw sanity check
  count: 108
  mean error: 0.000002001 deg
  max error: 0.000009713 deg


## 4. Batch Query Localization

In [ ]:
import time
import numpy as np

localization_results = []
inference_times_ms = []

# ── PRE-STEP: Copy all query images to cache directory at once ────────────────
# FIX: Moved outside the loop to avoid blocking I/O on every iteration.
print('Copying query images to cache...')
for query_path in query_paths:
    dst = query_cache_dir / query_path.name
    if not dst.exists():
        shutil.copy2(query_path, dst)

all_query_rels = [f'{query_cache_dir.name}/{p.name}' for p in query_paths]
print(f'  {len(all_query_rels)} images ready in cache.')

# Shared H5 feature/match files for all queries.
# FIX: Single shared H5 per feature type instead of per-image files.
all_query_global_features = results_dir / 'query-global-feats-netvlad.h5'
all_query_features        = results_dir / 'query-features.h5'
all_query_matches         = results_dir / 'query-matches.h5'

# ── STEP 1: Extract global features for ALL queries in one call ───────────────
# FIX: Model (NetVLAD) is loaded ONCE here instead of once per image.
print('\nExtracting global features (NetVLAD) for all queries...')
t0 = time.perf_counter()
extract_features.main(
    retrieval_conf,
    map_bundle_root,
    image_list=all_query_rels,
    feature_path=all_query_global_features,
    overwrite=overwrite_query_features,
)
print(f'  Done in {(time.perf_counter() - t0)*1000:.0f} ms')

# ── STEP 2: Extract local features for ALL queries in one call ────────────────
# FIX: Model (SuperPoint) is loaded ONCE here instead of once per image.
print('\nExtracting local features (SuperPoint) for all queries...')
t0 = time.perf_counter()
extract_features.main(
    feature_conf,
    map_bundle_root,
    image_list=all_query_rels,
    feature_path=all_query_features,
    overwrite=overwrite_query_features,
)
print(f'  Done in {(time.perf_counter() - t0)*1000:.0f} ms')

# ── STEP 3: Per-image retrieval pairing + matching + PnP ─────────────────────
# Features are already extracted; the loop only does lightweight work per image.
print('\nRunning per-image retrieval, matching, and pose estimation...')

for idx, (query_path, query_rel) in enumerate(zip(query_paths, all_query_rels), start=1):
    query_basename = query_path.name
    query_gt = pose_by_name[query_basename]
    query_gt_center = metadata_pose_center(query_gt)

    stem = Path(query_basename).stem
    loc_pairs = results_dir / f'{stem}-pairs-query-netvlad.txt'

    print(f'\n[{idx}/{len(query_paths)}] {query_basename} camera={query_gt["camera_name"]}')

    try:
        start_time = time.perf_counter()

        # Retrieval: find top-N database images for this query.
        pairs_from_retrieval.main(
            descriptors=all_query_global_features,
            output=loc_pairs,
            num_matched=min(num_loc, len(references)),
            query_list=[query_rel],
            db_list=references,
            db_descriptors=db_global_features,
        )

        # Match local features between query and retrieved database images.
        # FIX: LightGlue matcher is loaded once per call; batching all pairs
        #      for this query minimises that overhead.
        match_features.main(
            matcher_conf,
            loc_pairs,
            features=all_query_features,
            features_ref=db_features,
            matches=all_query_matches,
            overwrite=True,
        )

        retrieval_dict  = parse_retrieval(loc_pairs)
        retrieved_names = retrieval_dict[query_rel]
        ref_ids         = resolve_reference_ids(model, retrieved_names)

        ret, log = pose_from_cluster(
            localizer, query_rel, camera, ref_ids,
            all_query_features, all_query_matches,
        )

        end_time   = time.perf_counter()
        elapsed_ms = (end_time - start_time) * 1000.0

        if ret is None:
            print('  pose estimation failed')
            localization_results.append({
                'image_name':  query_basename,
                'camera_name': query_gt['camera_name'],
                'success':     False,
                'error':       'pose_from_cluster returned None',
                'retrieved':   retrieved_names,
            })
            continue

        R_wc_rh, tvec    = pycolmap_transform_rt(ret['cam_from_world'])
        estimated_center = -R_wc_rh.T @ tvec
        position_error_m = float(np.linalg.norm(estimated_center - query_gt_center))

        estimated_heading = colmap_world_to_camera_to_carla_yaw_deg(R_wc_rh)
        gt_heading        = float(query_gt['yaw_deg'])
        heading_error     = heading_error_deg(estimated_heading, gt_heading)

        inference_times_ms.append(elapsed_ms)

        result = {
            'image_name':           query_basename,
            'camera_name':          query_gt['camera_name'],
            'capture_id':           int(query_gt['capture_id']),
            'success':              True,
            'num_inliers':          int(ret['num_inliers']),
            'position_error_m':     position_error_m,
            'estimated_center_x':   float(estimated_center[0]),
            'estimated_center_y':   float(estimated_center[1]),
            'estimated_center_z':   float(estimated_center[2]),
            'gt_center_x':          float(query_gt_center[0]),
            'gt_center_y':          float(query_gt_center[1]),
            'gt_center_z':          float(query_gt_center[2]),
            'estimated_heading_deg': estimated_heading,
            'ground_truth_yaw_deg': gt_heading,
            'heading_error_deg':    float(heading_error),
            'retrieved':            retrieved_names,
            'inference_time_ms':    elapsed_ms,
        }
        localization_results.append(result)
        print(f"  success inliers={result['num_inliers']} pos_err={position_error_m:.3f} m "
              f"heading_err={heading_error:.2f} deg time={elapsed_ms:.2f} ms")

    except Exception as exc:
        print(f'  error: {exc}')
        localization_results.append({
            'image_name':  query_basename,
            'camera_name': query_gt['camera_name'],
            'capture_id':  int(query_gt['capture_id']),
            'success':     False,
            'error':       str(exc),
        })

print('\nBatch localization finished')

if inference_times_ms:
    mean_time = np.mean(inference_times_ms)
    std_time  = np.std(inference_times_ms)
    min_time  = np.min(inference_times_ms)
    max_time  = np.max(inference_times_ms)
    print('\n--- Inference Time Metrics (retrieval + matching + PnP per image) ---')
    print(f'Mean Inference Time: {mean_time:.2f} ms')
    print(f'Std Deviation:       {std_time:.2f} ms')
    print(f'Min / Max Time:      {min_time:.2f} ms / {max_time:.2f} ms')


[2026/05/24 16:10:06 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}


[2026/05/24 16:10:06 hloc INFO] Skipping the extraction.
[2026/05/24 16:10:06 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}



[1/108] 000001_front_f00009863.png camera=front
Loaded SuperPoint model


100%|██████████| 1/1 [00:02<00:00,  2.34s/it]
[2026/05/24 16:10:08 hloc INFO] Finished exporting features.
[2026/05/24 16:10:08 hloc INFO] Extracting image pairs from a retrieval database.
[2026/05/24 16:10:09 hloc INFO] Found 10 pairs.
[2026/05/24 16:10:09 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
c:\Users\ilker\Desktop\bitirme-project\venv38\lib\site-packages\kornia\feature\lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
  0%|          | 0/10 [00:00<?, ?it/s]c:\Users\ilker\Desktop\bitirme-project\venv38\lib\site-packages\lightglue\lightglue.py:120: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\na

  success inliers=2812 pos_err=0.028 m heading_err=0.14 deg

[2/108] 000002_front_f00009934.png camera=front


...

  success inliers=2206 pos_err=0.108 m heading_err=0.02 deg

[108/108] 000108_front_f00024071.png camera=front


100%|██████████| 1/1 [00:01<00:00,  1.17s/it]
[2026/05/24 16:27:06 hloc INFO] Finished exporting features.
[2026/05/24 16:27:06 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}


Loaded SuperPoint model


100%|██████████| 1/1 [00:02<00:00,  2.25s/it]
[2026/05/24 16:27:08 hloc INFO] Finished exporting features.
[2026/05/24 16:27:08 hloc INFO] Extracting image pairs from a retrieval database.
[2026/05/24 16:27:08 hloc INFO] Found 10 pairs.
[2026/05/24 16:27:08 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
100%|██████████| 10/10 [00:00<00:00, 12.33it/s]
[2026/05/24 16:27:09 hloc INFO] Finished exporting matches.


  success inliers=1448 pos_err=0.027 m heading_err=0.00 deg

Batch localization finished


## 5. Results Summary

In [9]:
results_df = pd.DataFrame(localization_results)
display(results_df)

successful = results_df[results_df['success'] == True]
failed = results_df[results_df['success'] != True]

summary = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'dataset_root': str(dataset_root),
    'query_source_dir': str(query_source_dir),
    'map_bundle_root': str(map_bundle_root),
    'sfm_model_root': str(sfm_model_root),
    'total_queries': int(len(results_df)),
    'successful_queries': int(len(successful)),
    'failed_queries': int(len(failed)),
    'success_rate': float(len(successful) / max(1, len(results_df))),
    'num_retrieved': num_loc,
    'ransac_max_error_px': max_error,
}

if len(successful) > 0:
    summary['position_error_m'] = {
        'mean': float(successful['position_error_m'].mean()),
        'median': float(successful['position_error_m'].median()),
        'max': float(successful['position_error_m'].max()),
    }
    summary['inliers'] = {
        'mean': float(successful['num_inliers'].mean()),
        'median': float(successful['num_inliers'].median()),
        'min': int(successful['num_inliers'].min()),
    }
    if 'heading_error_deg' in successful:
        summary['heading_error_deg'] = {
            'mean': float(successful['heading_error_deg'].mean()),
            'median': float(successful['heading_error_deg'].median()),
            'max': float(successful['heading_error_deg'].max()),
        }

print(json.dumps(summary, indent=2))

details_csv = results_dir / 'query_batch_details.csv'
summary_json = results_dir / 'query_batch_summary.json'
results_df.to_csv(details_csv, index=False)
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Details CSV: {details_csv}')
print(f'Summary JSON: {summary_json}')

,image_name,camera_name,capture_id,success,num_inliers,position_error_m,estimated_center_x,estimated_center_y,estimated_center_z,gt_center_x,gt_center_y,gt_center_z,estimated_heading_deg,ground_truth_yaw_deg,heading_error_deg,retrieved
0,000001_front_f00009863.png,front,1,True,2812,0.027528,-4.858079,-2.163367,9.883810,-4.885180,-2.167644,9.886053,104.995956,105.134682,0.138725,"[images/000059_front_f00007839.png, images/000..."
1,000002_front_f00009934.png,front,2,True,1369,0.056124,0.166685,-2.197539,11.316845,0.212937,-2.201335,11.348409,82.060521,82.105362,0.044841,"[images/000418_front_f00030357.png, images/000..."
2,000003_front_f00010004.png,front,3,True,1963,0.016881,7.310510,-2.192419,7.302847,7.320307,-2.200959,7.313620,123.934780,123.977325,0.042546,"[images/000065_front_f00008166.png, images/000..."
3,000004_front_f00010092.png,front,4,True,3594,0.012920,17.037124,-2.199846,0.756175,17.044885,-2.200731,0.745884,113.630770,113.623260,0.007509,"[images/000081_front_f00008999.png, images/000..."
4,000005_front_f00010178.png,front,5,True,3116,0.080702,25.988805,-2.200825,-3.059061,26.064013,-2.201399,-3.088324,112.457503,112.456123,0.001379,"[images/000090_front_f00009428.png, images/000..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,000104_front_f00023657.png,front,104,True,1350,0.173106,12.700512,-2.228350,29.466052,12.746348,-2.224928,29.632944,16.677131,16.673391,0.003740,"[images/000428_front_f00031038.png, images/000..."
104,000105_front_f00023747.png,front,105,True,1467,0.023262,18.782343,-2.205914,42.557880,18.787741,-2.201626,42.580097,26.730722,26.739618,0.008897,"[images/000435_front_f00031854.png, images/000..."
105,000106_front_f00023852.png,front,106,True,1738,0.007585,24.240497,-2.203837,52.209763,24.238360,-2.201675,52.202814,29.127933,29.133177,0.005244,"[images/000441_front_f00032477.png, images/000..."
106,000107_front_f00023975.png,front,107,True,2206,0.108131,30.063687,-2.205000,67.594486,30.087997,-2.201357,67.699786,10.431274,10.416160,0.015114,"[images/000454_front_f00033415.png, images/000..."


{
  "timestamp": "2026-05-24T16:27:10",
  "dataset_root": "..\\datasets\\test_dataset_108_24_may",
  "query_source_dir": "..\\datasets\\test_dataset_108_24_may\\test",
  "map_bundle_root": "..\\outputs\\train_dataset_585_24_may-bundle",
  "sfm_model_root": "..\\outputs\\train_dataset_585_24_may-bundle\\sfm",
  "total_queries": 108,
  "successful_queries": 108,
  "failed_queries": 0,
  "success_rate": 1.0,
  "num_retrieved": 10,
  "ransac_max_error_px": 12,
  "position_error_m": {
    "mean": 0.2058140753712587,
    "median": 0.042237216143365106,
    "max": 15.238748478691798
  },
  "inliers": {
    "mean": 1517.7407407407406,
    "median": 1443.0,
    "min": 130
  },
  "heading_error_deg": {
    "mean": 0.24103370389217446,
    "median": 0.01196064654284612,
    "max": 22.91525806497623
  }
}
Details CSV: ..\outputs\train_dataset_585_24_may-bundle\query_batch_results_v2_24_may\query_batch_details.csv
Summary JSON: ..\outputs\train_dataset_585_24_may-bundle\query_batch_results_v2_24_ma

## 6. VisualLocalization.net Threshold Format

In [10]:
benchmark_thresholds = [
    (0.25, 2.0),
    (0.50, 5.0),
    (5.00, 10.0),
]

total_queries = len(results_df)
successful_eval = successful.dropna(subset=['position_error_m', 'heading_error_deg']).copy()

benchmark_parts = []
benchmark_rows = []
for pos_thr, rot_thr in benchmark_thresholds:
    passed = successful_eval[
        (successful_eval['position_error_m'] <= pos_thr)
        & (successful_eval['heading_error_deg'] <= rot_thr)
    ]
    count = int(len(passed))
    percent_total = 100.0 * count / max(1, total_queries)
    percent_successful = 100.0 * count / max(1, len(successful_eval))
    benchmark_parts.append(f'{percent_total:.1f}')
    benchmark_rows.append({
        'position_threshold_m': pos_thr,
        'heading_threshold_deg': rot_thr,
        'passed': count,
        'total_queries': int(total_queries),
        'successful_evaluated_queries': int(len(successful_eval)),
        'percent_of_all_queries': percent_total,
        'percent_of_successful_queries': percent_successful,
    })

benchmark_df = pd.DataFrame(benchmark_rows)
print('VisualLocalization.net style localization scores')
print('All conditions: (0.25m, 2 deg) / (0.5m, 5 deg) / (5m, 10 deg)')
print('All conditions: ' + ' / '.join(benchmark_parts))
display(benchmark_df)

benchmark_json = results_dir / 'query_batch_visual_localization_thresholds.json'
benchmark_json.write_text(json.dumps(benchmark_rows, indent=2), encoding='utf-8')
print(f'Benchmark thresholds JSON: {benchmark_json}')

VisualLocalization.net style localization scores
All conditions: (0.25m, 2 deg) / (0.5m, 5 deg) / (5m, 10 deg)
All conditions: 96.3 / 99.1 / 99.1


,position_threshold_m,heading_threshold_deg,passed,total_queries,successful_evaluated_queries,percent_of_all_queries,percent_of_successful_queries
0,0.25,2.0,104,108,108,96.296296,96.296296
1,0.50,5.0,107,108,108,99.074074,99.074074
2,5.00,10.0,107,108,108,99.074074,99.074074


Benchmark thresholds JSON: ..\outputs\train_dataset_585_24_may-bundle\query_batch_results_v2_24_may\query_batch_visual_localization_thresholds.json


## 7. Worst Successful Queries

In [11]:
if len(successful) > 0:
    worst = successful.sort_values('position_error_m', ascending=False).head(10)
    display(worst[['image_name', 'camera_name', 'capture_id', 'num_inliers', 'position_error_m', 'heading_error_deg']])
else:
    print('No successful queries to inspect.')

if len(failed) > 0:
    print('Failed queries:')
    display(failed[['image_name', 'camera_name', 'capture_id', 'error']])

,image_name,camera_name,capture_id,num_inliers,position_error_m,heading_error_deg
30,000031_front_f00013980.png,front,31,164,15.238748,22.915258
5,000006_front_f00010304.png,front,6,1941,0.357704,0.013127
86,000087_front_f00021692.png,front,87,1136,0.307201,0.066383
83,000084_front_f00021272.png,front,84,199,0.284148,0.676066
97,000098_front_f00023113.png,front,98,2974,0.237491,0.021827
77,000078_front_f00020588.png,front,78,1047,0.225302,0.063068
54,000055_front_f00017202.png,front,55,2066,0.180429,0.000517
47,000048_front_f00016411.png,front,48,1487,0.174705,0.002403
49,000050_front_f00016651.png,front,50,1115,0.173112,0.008635
103,000104_front_f00023657.png,front,104,1350,0.173106,0.003740
